In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

In [3]:
def load_instance_xlsx(path: str):
    meta = pd.read_excel(path, sheet_name="meta")
    meta = dict(zip(meta["parameter"], meta["value"]))

    concepts = pd.read_excel(path, sheet_name="concepts")
    # ids pueden no ser 1..N si quieres; aquí lo soportamos igual
    J = sorted(concepts["id"].astype(int).tolist())
    N = len(J)

    h = int(meta["h"])
    alpha = float(meta["alpha"])
    beta = float(meta["beta"])
    v = float(meta["v"])

    p = dict(zip(concepts["id"].astype(int), concepts["p"].astype(int)))
    w = dict(zip(concepts["id"].astype(int), concepts["w"].astype(float)))

    edges = pd.read_excel(path, sheet_name="precedence")
    if len(edges) == 0:
        A = []
    else:
        A = list(map(tuple, edges[["i","j"]].astype(int).to_numpy()))

    # Validaciones básicas
    assert all(j in p for j in J)
    assert all(j in w for j in J)
    assert all(p[j] >= 0 and p[j] <= h for j in J)
    assert all(i in p and j in p for (i,j) in A)

    # Conjuntos de tiempos factibles
    T = {j: range(0, h - p[j] + 1) for j in J}

    return {
        "N": N,
        "J": J,
        "p": p,
        "w": w,
        "h": h,
        "alpha": alpha,
        "beta": beta,
        "v": v,
        "A": A,
        "T": T,
        "concepts_df": concepts,   # opcional por si quieres nombres
    }

# uso
data = load_instance_xlsx("C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Intermedias\\instance_template_v2.xlsx")

N = data["N"]
J = data["J"]
p = data["p"]
w = data["w"]
h = data["h"]
alpha = data["alpha"]
beta = data["beta"]
v = data["v"]
A = data["A"]
T = data["T"]

In [4]:
# ==========================================
# 2) f(t) y su primitiva para integrar exacto
# ==========================================
def f_piecewise(t: float) -> float:
    """f(t) de la ecuación (4.6)."""
    if 0 <= t <= v:
        return alpha - (alpha / v) * t
    elif v < t <= h:
        return (beta / v) * t - beta
    else:
        return 0.0

def F_primitive(t: float) -> float:
    """
    Primitiva de f(t), truncando fuera de [0,h]
    para poder calcular integrales exactas como F(b)-F(a).
    """
    if t <= 0:
        return 0.0
    if t <= v:
        # ∫ (alpha - alpha/v * s) ds = alpha*s - alpha/(2v)*s^2
        return alpha * t - (alpha / (2.0 * v)) * t * t
    else:
        # + ∫_v^t (beta/v * s - beta) ds +constante para continuidad
        return (beta / (2.0 * v)) * t * t - beta * t + (v*(alpha + beta))/2.0

def integral_f(a: float, b: float) -> float:
    """∫_a^b f(u)du exacta."""
    return F_primitive(b) - F_primitive(a)


In [5]:
# 3) Coeficientes del objetivo: q_{j,t} = w_j * ∫_t^{t+p_j} f
q = {(j, t): w[j] * integral_f(t, t + p[j]) for j in J for t in T[j]}

In [ ]:
# =====================
# 4) Construcción MILP
# =====================
m = gp.Model("time_indexed_q_model")
m.Params.OutputFlag = 1  # cambia a 0 si no quieres logs

# Variables binarias x_{j,t} = 1 si actividad j inicia en t
x = m.addVars(((j, t) for j in J for t in T[j]), vtype=GRB.BINARY, name="x")

# (3.2) Maximizar sum_j sum_t q_{j,t} x_{j,t}
m.setObjective(gp.quicksum(q[j, t] * x[j, t] for j in J for t in T[j]), GRB.MAXIMIZE)

# (3.3) Presupuesto total de tiempo
m.addConstr(
    gp.quicksum(p[j] * x[j, t] for j in J for t in T[j]) <= h,
    name="time_budget"
)

# (3.4) Cada actividad a lo más una vez
for j in J:
    m.addConstr(
        gp.quicksum(x[j, t] for t in T[j]) <= 1,
        name=f"at_most_once[{j}]"
    )

# (3.5) No solapamiento
m.addConstrs(
    (
        gp.quicksum(
            x[j, tp]
            for j in J
            for tp in range(
                max(0, t - p[j] + 1),
                min(h - p[j] + 1, t + 1)
            )
        ) <= 1
        for t in range(0, h)
    ),
    name="capacity"
)


# (3.6) Precedencias Flexible
# x_{j,t} <= sum_{t'=0}^{t-p_i} x_{i,t'}   ∀(i,j)∈A, ∀t∈T_j ∩ [p_i, h-p_j]
m.addConstrs(
    (
        x[j,t] <= gp.quicksum(x[i,tp] for tp in T[i] if tp + p[i] <= t)
        for (i,j) in A
        for t in T[j]), 
        name="precedence_flexible"
    
)

# (3.6) Preferencia Estricta
#m.addConstrs(
#    (
#        gp.quicksum((t+p[i]) * x[i,t] for t in T[i]) <=
#        gp.quicksum(tp * x[j,tp] for tp in T[j])
#        for (i,j) in A
#        
#    ),
#    name="precedence_strict"
#)



m.update()

Set parameter Username
Set parameter LicenseID to value 2739745
Set parameter OutputFlag to value 1


In [7]:
m.update()
print("Vars:", m.NumVars, "Constr:", m.NumConstrs, "NZ:", m.NumNZs)

Vars: 3026 Constr: 7351 NZ: 311210


In [8]:
m.update()
m.write("modelo.mps")   # o "modelo.lp"

In [9]:

from highspy import Highs


r = Highs()
r.readModel("modelo.mps")
r.setOptionValue("output_flag", True)      # mostrar log
r.setOptionValue("log_to_console", True)   # si tu build lo soporta
r.setOptionValue("time_limit", 600.0)      # 5 minutos, ajusta
r.setOptionValue("mip_rel_gap", 0.05)      # 1% gap (ajusta)
r.setOptionValue("threads", 8)             # ajusta a tu CPU
r.run()



<HighsStatus.kOk: 0>

In [10]:
print("status:", r.getModelStatus())
info = r.getInfo()
print(info)

status: HighsModelStatus.kOptimal


In [11]:
obj = r.getObjectiveValue()
print("Obj:", obj)

Obj: 150.40179086155


In [12]:
info = r.getInfo()
print([a for a in dir(info) if "time" in a.lower() or "mip" in a.lower() or "gap" in a.lower() or "node" in a.lower() or "iter" in a.lower()])

['crossover_iteration_count', 'ipm_iteration_count', 'mip_dual_bound', 'mip_gap', 'mip_node_count', 'pdlp_iteration_count', 'qp_iteration_count', 'simplex_iteration_count']


In [13]:
fields = [
    "run_time", "mip_gap", "mip_node_count", "mip_dual_bound",
    "simplex_iteration_count", "ipm_iteration_count",
]
for f in fields:
    if hasattr(info, f):
        print(f, "=", getattr(info, f))

mip_gap = 0.0
mip_node_count = 1
mip_dual_bound = 150.40179086155
simplex_iteration_count = 270
ipm_iteration_count = -1


In [14]:
sol = r.getSolution()

# número de columnas (según versión)
if hasattr(r, "getNumCol"):
    ncol = r.getNumCol()
elif hasattr(r, "getNumCols"):
    ncol = r.getNumCols()
else:
    # fallback: desde el LP si existe
    lp = r.getLp()
    ncol = getattr(lp, "num_col_", None) or getattr(lp, "num_col", None)

col_names = []
for i in range(ncol):
    status, name = r.getColName(i)   # <- devuelve (HighsStatus, "nombre")
    col_names.append(name)

x_val = dict(zip(col_names, sol.col_value))
chosen = {name: val for name, val in x_val.items() if val > 0.5}

list(chosen.items())[:20]

[('x[1,20]', 1.0),
 ('x[2,27]', 0.9999999999999882),
 ('x[3,36]', 0.9999999999999882),
 ('x[4,66]', 0.9999999999999858),
 ('x[15,0]', 1.0),
 ('x[16,41]', 0.9999999999999765),
 ('x[17,71]', 0.999999999999985),
 ('x[30,60]', 0.9999999999999769),
 ('x[31,53]', 0.9999999999999906)]

In [15]:
import re

pat = re.compile(r"x\[(\d+),(\d+)\]")

starts = []
for name, val in chosen.items():
    m = pat.match(name)
    if m:
        j = int(m.group(1))
        t = int(m.group(2))
        starts.append((t, j))

starts.sort()  # orden por tiempo
starts

[(0, 15),
 (20, 1),
 (27, 2),
 (36, 3),
 (41, 16),
 (53, 31),
 (60, 30),
 (66, 4),
 (71, 17)]

In [16]:
#p = data["p"]   # <-- define p aquí
schedule = []
for t, j in starts:
    schedule.append({
        "j": j,
        "start": t,
        "duration": int(p[j]),
        "end": t + int(p[j])
    })

sched_df = pd.DataFrame(schedule).sort_values("start").reset_index(drop=True)
sched_df

,j,start,duration,end
0,15,0,20,20
1,1,20,7,27
2,2,27,9,36
3,3,36,5,41
4,16,41,12,53
5,31,53,7,60
6,30,60,6,66
7,4,66,5,71
8,17,71,19,90


In [17]:
concepts = data["concepts_df"][["id","name"]].copy()
concepts["id"] = concepts["id"].astype(int)

sched_df = sched_df.merge(concepts, left_on="j", right_on="id", how="left") \
                   .drop(columns=["id"]) \
                   .rename(columns={"name":"concept"})

sched_df

,j,start,duration,end,concept
0,15,0,20,20,Modalidades_Pensiones
1,1,20,7,27,Que_es_pension
2,2,27,9,36,Pension_Autofinanciada
3,3,36,5,41,AFP
4,16,41,12,53,Riesgo de longevidad
5,31,53,7,60,Pension_Sobrevivencia
6,30,60,6,66,Beneficiarios_Legales
7,4,66,5,71,Compania_Seguros
8,17,71,19,90,Riesgo de rentabilidad


In [ ]:
sol = r.getSolution()
lp = r.getLp()

# valores por variable (en el mismo orden que lp.col_names)
x_val = dict(zip(lp.col_names, sol.col_value))

# ejemplo: ver variables que quedaron en 1 (si son binarias)
chosen = {name: val for name, val in x_val.items() if val > 0.5}
list(chosen.items())[:20]

In [ ]:
# 5) Resolver

m.optimize()

In [ ]:
# =====================
# 6) Reporte de solución
# =====================
if m.Status == GRB.OPTIMAL:
    print("\n=== SOLUCIÓN ÓPTIMA ===")
    print(f"Valor objetivo = {m.ObjVal:.6f}")

    selected = []
    for j in J:
        for t in T[j]:
            if x[j, t].X > 0.5:
                selected.append((j, t, t + p[j], p[j], w[j], q[j, t]))

    selected.sort(key=lambda z: z[1])

    print("\nActividades seleccionadas (ordenadas por inicio):")
    for (j, start, end, dur, wj, qjt) in selected:
        print(f"  Act {j}: inicio={start:3d}, fin={end:3d}, p={dur:3d}, w={wj:3d}, q_jt={qjt:.6f}")

    selected_ids = {j for (j, *_rest) in selected}
    omitted = [j for j in J if j not in selected_ids]
    print(f"\nActividades NO seleccionadas: {omitted}")

    # Mostrar huecos (idle), por si te interesa
    print("\nHuecos (idle):")
    current = 0
    for (_, start, end, *_rest) in selected:
        if start > current:
            print(f"  [{current}, {start})  duración={start-current}")
        current = end
    if current < h:
        print(f"  [{current}, {h})  duración={h-current}")

elif m.Status == GRB.INFEASIBLE:
    print("El modelo es infactible.")
else:
    print(f"Estado del solver: {m.Status}")